# Python Collections Module Exercises: 25 Coding Problems with Solutions

A practice notebook on `Counter`, `defaultdict`, `OrderedDict`, `deque`, and `namedtuple` — frequency analysis, grouping, LRU caching, queues/stacks, and lightweight records — each with a concept note, a hint, a solution, and an explanation.

*Adapted for practice from the exercise list at [PYnative](https://pynative.com/python-collections-module-exercises/).*

---

## Concepts you'll need

This set covers Python's `collections` module — five purpose-built alternatives to the plain `dict`, `list`, and `tuple`.

- **`Counter`** — a dict subclass for tallying items. `Counter(iterable)` counts in one call; `.most_common(n)` ranks by frequency; `+`, `-` do element-wise arithmetic (`-` drops zero/negative results, `.subtract()` keeps them).
- **`defaultdict(factory)`** — a dict that auto-initializes a missing key by calling `factory()` (e.g. `list`, `int`, `set`), eliminating "if key not in dict" boilerplate before appending/incrementing. `defaultdict(lambda: defaultdict(list))` nests two levels, each auto-initializing independently.
- **`OrderedDict`** — like a dict but with order-aware extras: `.move_to_end(key, last=True/False)` repositions a key in O(1); `reversed(od.items())` iterates backwards; equality between two `OrderedDict`s is order-sensitive (unlike plain dicts, where `{"a":1,"b":2} == {"b":2,"a":1}` is `True`).
- **`deque`** — a double-ended queue with O(1) operations at *both* ends: `.append()`/`.pop()` (right), `.appendleft()`/`.popleft()` (left) — unlike a list, where inserting/removing at index 0 is O(n). `.rotate(n)` shifts elements circularly; `deque(maxlen=N)` auto-evicts from the opposite end once full, perfect for "last N items" buffers.
- **`namedtuple`** — a tuple subclass with named fields: `Point = namedtuple("Point", ["x", "y"])` lets you write `p.x` instead of `p[0]`. Still fully immutable and index-compatible. `._asdict()` converts to a dict (for JSON, etc.), `._replace(field=value)` returns a *new* instance with one field changed, and `defaults=[...]` gives trailing fields fallback values.

Each exercise below gives a problem, a hint, a solution, and an explanation.

## Exercise 1. Word Frequency Counter

**Concept:** Counter(iterable)

**Problem:** Count how many times each word appears in a sentence.

**Given:**
```
sentence = "the cat sat on the mat the cat sat"
```

**Expected Output:**
```
the: 3
cat: 2
sat: 2
on: 1
mat: 1
```

**Hint:** Counter(words) builds the whole frequency map in one call — no manual loop needed.

In [ ]:
from collections import Counter

sentence = "the cat sat on the mat the cat sat"
words = sentence.lower().split()

word_count = Counter(words)

for word, count in word_count.items():
    print(f"{word}: {count}")

**Explanation:** Counter accepts any iterable and returns a dict subclass mapping each unique element to its count. Lowercasing before splitting ensures 'The' and 'the' are counted as one word. Since Counter is a dict subclass, .items() works exactly as it would on a plain dictionary.

## Exercise 2. Most Common Elements

**Concept:** Counter.most_common(n)

**Problem:** Find the top 3 most frequently occurring items in a list.

**Given:**
```
items = ["apple", "banana", "apple", "cherry", "banana", "apple", "date", "cherry", "banana", "apple"]
```

**Expected Output:**
```
apple: 4
banana: 3
cherry: 2
```

**Hint:** counter.most_common(3) returns the top 3 as (element, count) tuples, already sorted.

In [ ]:
from collections import Counter

items = ["apple", "banana", "apple", "cherry", "banana", "apple",
         "date", "cherry", "banana", "apple"]

item_count = Counter(items)

print("Top 3 most common elements:")
for item, count in item_count.most_common(3):
    print(f"  {item}: {count}")

**Explanation:** most_common(3) returns the 3 highest-count entries sorted from most to least frequent, internally using heapq.nlargest for efficiency. Unpacking each tuple directly into item, count reads more cleanly than indexing into pair[0] and pair[1].

## Exercise 3. Subtract Two Counters

**Concept:** Counter subtraction (-) vs. .subtract()

**Problem:** Find remaining stock after sales, comparing the - operator against .subtract().

**Given:**
```
stock = ["apple","apple","apple","banana","banana","cherry"], sold = ["apple","apple","banana","cherry","cherry"]
```

**Expected Output:**
```
Remaining (positive only): apple: 1, banana: 1
After .subtract(): includes zero/negative counts too
```

**Hint:** - drops zero-or-below results automatically; .subtract() keeps them and modifies in place.

In [ ]:
from collections import Counter

stock = ["apple", "apple", "apple", "banana", "banana", "cherry"]
sold  = ["apple", "apple", "banana", "cherry", "cherry"]

stock_counter = Counter(stock)
sold_counter  = Counter(sold)

print("Stock :", dict(stock_counter))
print("Sold  :", dict(sold_counter))

remaining = stock_counter - sold_counter

print("\nRemaining stock (positive counts only):")
for item, count in remaining.items():
    print(f"  {item}: {count}")

stock_counter.subtract(sold_counter)
print("\nAfter .subtract() (includes zero and negative):")
for item, count in stock_counter.items():
    print(f"  {item}: {count}")

**Explanation:** stock_counter - sold_counter builds a brand-new Counter containing only items whose count came out strictly positive — cherry drops out entirely since 1 - 2 = -1. .subtract() instead mutates stock_counter in place and keeps every result, including zero and negative counts, which is useful for spotting overselling.

## Exercise 4. Character Frequency in a String

**Concept:** Counter on a string, filtered and sorted

**Problem:** Count character frequencies in a string, showing only letters, sorted by frequency.

**Given:**
```
text = "hello world"
```

**Expected Output:**
```
l: 3
o: 2
h: 1
e: 1
w: 1
r: 1
d: 1
```

**Hint:** Counter(text) treats each character as an element; filter with char.isalpha() while iterating .most_common().

In [ ]:
from collections import Counter

text = "hello world"

char_count = Counter(text)

print("Alphabetic character frequencies (most to least):")
for char, count in char_count.most_common():
    if char.isalpha():
        print(f"  '{char}': {count}")

**Explanation:** Passing a string directly to Counter() counts each individual character, space included. most_common() with no argument returns every element ranked highest to lowest; the isalpha() check inside the loop then filters out the space without disturbing that frequency order.

## Exercise 5. Combine Counters With Addition

**Concept:** Counter + Counter

**Problem:** Merge two inventory Counters into a single combined total.

**Given:**
```
warehouse_a = Counter({"apple": 50, "banana": 30, "cherry": 20}), warehouse_b = Counter({"banana": 40, "cherry": 10, "date": 60})
```

**Expected Output:**
```
apple: 50
banana: 70
cherry: 30
date: 60
```

**Hint:** + adds counts for shared keys and carries over keys unique to either side, producing a new Counter.

In [ ]:
from collections import Counter

warehouse_a = Counter({"apple": 50, "banana": 30, "cherry": 20})
warehouse_b = Counter({"banana": 40, "cherry": 10, "date": 60})

combined = warehouse_a + warehouse_b

print("Merged inventory:")
for item, count in sorted(combined.items()):
    print(f"  {item}: {count}")

**Explanation:** + adds matching keys together (banana: 30 + 40 = 70) and simply includes keys present in only one counter (date: 60), all without modifying either original — unlike .update(), which would change a counter in place. sorted(combined.items()) then gives predictable alphabetical output regardless of insertion order.

## Exercise 6. Group Words by First Letter

**Concept:** defaultdict(list)

**Problem:** Group a list of words by their starting letter.

**Given:**
```
words = ["apple", "avocado", "banana", "blueberry", "cherry", "apricot", "cranberry", "bluebell"]
```

**Expected Output:**
```
a: ['apple', 'avocado', 'apricot']
b: ['banana', 'blueberry', 'bluebell']
c: ['cherry', 'cranberry']
```

**Hint:** grouped[key].append(word) works immediately, since defaultdict(list) auto-creates an empty list for any new key.

In [ ]:
from collections import defaultdict

words = ["apple", "avocado", "banana", "blueberry",
         "cherry", "apricot", "cranberry", "bluebell"]

grouped = defaultdict(list)

for word in words:
    key = word[0].lower()
    grouped[key].append(word)

for letter, group in sorted(grouped.items()):
    print(f"{letter}: {group}")

**Explanation:** defaultdict(list) removes the usual 'if key not in d: d[key] = []' boilerplate — accessing a missing key automatically creates an empty list for it before .append() runs. sorted(grouped.items()) displays the groups in alphabetical order, since defaultdict otherwise follows the same insertion-order iteration as a plain dict.

## Exercise 7. Count Occurrences Without KeyError

**Concept:** defaultdict(int)

**Problem:** Count occurrences of each item in a list, using defaultdict(int) instead of manual key checks.

**Given:**
```
colours = ["red", "blue", "red", "green", "blue", "blue", "red", "yellow"]
```

**Expected Output:**
```
red: 3
blue: 3
green: 1
yellow: 1
```

**Hint:** counts[item] += 1 works safely from the very first access, since int() (the factory) returns 0.

In [ ]:
from collections import defaultdict

colours = ["red", "blue", "red", "green", "blue", "blue", "red", "yellow"]

counts = defaultdict(int)

for colour in colours:
    counts[colour] += 1

print("Colour counts:")
for colour, count in counts.items():
    print(f"  {colour}: {count}")

**Explanation:** The argument int is a callable, invoked with no arguments the first time a missing key is accessed; since int() evaluates to 0, every new key silently starts at zero, making += 1 safe immediately with no KeyError risk. Counter is essentially a specialized defaultdict(int) with extra methods like .most_common() layered on top.

## Exercise 8. Build an Adjacency List

**Concept:** defaultdict(list) for graph representation

**Problem:** Build a directed graph's adjacency list from a list of edge pairs.

**Given:**
```
edges = [("A","B"), ("A","C"), ("B","D"), ("C","D"), ("D","E")]
```

**Expected Output:**
```
A: ['B', 'C']
B: ['D']
C: ['D']
D: ['E']
```

**Hint:** graph[src].append(dst) for a directed graph; add graph[dst].append(src) too for an undirected one.

In [ ]:
from collections import defaultdict

edges = [("A", "B"), ("A", "C"), ("B", "D"), ("C", "D"), ("D", "E")]

graph = defaultdict(list)

for src, dst in edges:
    graph[src].append(dst)

print("Adjacency list (directed graph):")
for node, neighbours in sorted(graph.items()):
    print(f"  {node}: {neighbours}")

undirected = defaultdict(list)
for src, dst in edges:
    undirected[src].append(dst)
    undirected[dst].append(src)

print("\nAdjacency list (undirected graph):")
for node, neighbours in sorted(undirected.items()):
    print(f"  {node}: {neighbours}")

**Explanation:** Unpacking each edge tuple directly into src, dst is cleaner than indexing edge[0]/edge[1]. In the directed version, one .append() per edge suffices since an edge A->B doesn't imply B->A; the undirected version simply adds both directions with a second .append() call.

## Exercise 9. Nested defaultdict

**Concept:** defaultdict(lambda: defaultdict(list))

**Problem:** Build a two-level department -> employee -> [] structure with no manual inner-dict setup.

**Given:**
```
records = [("Engineering","Alice"), ("Marketing","Bob"), ("Engineering","Charlie"), ("Marketing","Diana"), ("HR","Eve")]
```

**Expected Output:**
```
Engineering:
  - Alice
  - Charlie
HR:
  - Eve
Marketing:
  - Bob
  - Diana
```

**Hint:** A lambda default factory is required so each outer key gets its own separate inner defaultdict(list), not a shared one.

In [ ]:
from collections import defaultdict

records = [
    ("Engineering", "Alice"),
    ("Marketing",   "Bob"),
    ("Engineering", "Charlie"),
    ("Marketing",   "Diana"),
    ("HR",          "Eve"),
]

org = defaultdict(lambda: defaultdict(list))

for dept, employee in records:
    org[dept][employee]

print("Organisation structure:")
for dept, employees in sorted(org.items()):
    print(f"  {dept}:")
    for name in sorted(employees):
        print(f"    - {name}")

**Explanation:** Using a lambda as the factory means each new outer key gets a freshly created inner defaultdict(list) of its own; passing defaultdict(list) directly (without the lambda wrapper) would instead share one single inner dict across every department, which is not the intended behavior. Accessing org[dept][employee] triggers both levels' auto-initialization in sequence, with no explicit if-checks needed at either level.

## Exercise 10. defaultdict With set

**Concept:** defaultdict(set) for automatic deduplication

**Problem:** Collect each student's unique enrolled subjects, ignoring duplicate (student, subject) pairs.

**Given:**
```
enrolments = [("Alice","Maths"), ("Bob","Science"), ("Alice","Science"), ("Alice","Maths"), ("Bob","Maths"), ("Charlie","Science")]
```

**Expected Output:**
```
Alice: ['Maths', 'Science']
Bob: ['Maths', 'Science']
Charlie: ['Science']
```

**Hint:** .add() (not .append()) is the set method for insertion; adding a duplicate value is a silent no-op.

In [ ]:
from collections import defaultdict

enrolments = [
    ("Alice",   "Maths"),
    ("Bob",     "Science"),
    ("Alice",   "Science"),
    ("Alice",   "Maths"),
    ("Bob",     "Maths"),
    ("Charlie", "Science"),
]

subjects_by_student = defaultdict(set)

for student, subject in enrolments:
    subjects_by_student[student].add(subject)

print("Student enrolments (unique subjects):")
for student, subjects in sorted(subjects_by_student.items()):
    print(f"  {student}: {sorted(subjects)}")

**Explanation:** defaultdict(set) auto-creates an empty set() for a new student on first access. .add() is the set's insertion method — attempting to add an already-present value changes nothing and raises no error, which is exactly why the repeated ('Alice','Maths') pair has no visible effect. sorted(subjects) gives predictable alphabetical display since sets carry no inherent order.

## Exercise 11. Preserve Insertion Order

**Concept:** OrderedDict basics

**Problem:** Confirm that an OrderedDict's iteration order matches its construction order, and stays that way after reads.

**Given:**
```
pairs = [("banana",3), ("apple",5), ("cherry",1), ("date",8), ("elderberry",2)]
```

**Expected Output:**
```
banana: 3
apple: 5
cherry: 1
date: 8
elderberry: 2 (unchanged after reading 'apple')
```

**Hint:** OrderedDict(pairs) stores pairs in exactly the order given; reading a key never moves it.

In [ ]:
from collections import OrderedDict

pairs = [("banana", 3), ("apple", 5), ("cherry", 1),
         ("date", 8), ("elderberry", 2)]

od = OrderedDict(pairs)

print("Insertion order preserved:")
for key, value in od.items():
    print(f"  {key}: {value}")

_ = od["apple"]

print("\nOrder after accessing 'apple' (unchanged):")
for key, value in od.items():
    print(f"  {key}: {value}")

**Explanation:** OrderedDict accepts any iterable of (key, value) pairs and preserves their exact given order. Reading a value with od["apple"] does not reposition that key — only explicit operations like move_to_end() or a delete-then-reinsert would change the order.

## Exercise 12. Move to End

**Concept:** OrderedDict.move_to_end(key, last=True/False)

**Problem:** Promote one task to the front and demote another to the back of an ordered task list.

**Given:**
```
5 tasks in an OrderedDict
```

**Expected Output:**
```
task_3 moves to front; task_1 moves to back
```

**Hint:** move_to_end(key, last=False) moves to the front; move_to_end(key) (default last=True) moves to the back.

In [ ]:
from collections import OrderedDict

tasks = OrderedDict([
    ("task_1", "Write tests"),
    ("task_2", "Fix bug"),
    ("task_3", "Deploy"),
    ("task_4", "Code review"),
    ("task_5", "Update docs"),
])

def print_tasks(label, od):
    print(f"\n{label}:")
    for key, value in od.items():
        print(f"  {key}: {value}")

print_tasks("Original order", tasks)

tasks.move_to_end("task_3", last=False)
print_tasks("After moving 'task_3' to front", tasks)

tasks.move_to_end("task_1")
print_tasks("After moving 'task_1' to back", tasks)

**Explanation:** move_to_end() repositions an existing key in O(1) time, thanks to OrderedDict's internal doubly-linked list. last=False sends the key to the front (useful for marking something urgent); omitting last (defaulting to True) sends it to the back instead — both without touching any of the other keys' relative order.

## Exercise 13. Implement a Simple LRU Cache

**Concept:** OrderedDict as the backbone of an LRU cache

**Problem:** Build a fixed-capacity cache that evicts the least-recently-used entry when it fills up.

**Given:**
```
capacity = 3, then a sequence of get/put calls
```

**Expected Output:**
```
Cache fills, then evicts the oldest untouched entry once capacity is exceeded
```

**Hint:** move_to_end(key) on every get marks recency; popitem(last=False) on a full cache evicts the least-recent entry.

In [ ]:
from collections import OrderedDict

class LRUCache:
    def __init__(self, capacity):
        self.capacity = capacity
        self.cache = OrderedDict()

    def get(self, key):
        if key not in self.cache:
            return -1
        self.cache.move_to_end(key)
        return self.cache[key]

    def put(self, key, value):
        if key in self.cache:
            self.cache.move_to_end(key)
        self.cache[key] = value
        if len(self.cache) > self.capacity:
            evicted_key, evicted_val = self.cache.popitem(last=False)
            print(f"  [evicted] {evicted_key}: {evicted_val}")

    def display(self):
        print(f"  Cache (LRU to MRU): {list(self.cache.items())}")

lru = LRUCache(capacity=3)

print("put(A, 1)"); lru.put("A", 1); lru.display()
print("put(B, 2)"); lru.put("B", 2); lru.display()
print("put(C, 3)"); lru.put("C", 3); lru.display()
print(f"get(A) -> {lru.get('A')}");  lru.display()
print("put(D, 4) -- eviction expected")
lru.put("D", 4); lru.display()
print(f"get(B) -> {lru.get('B')}")

**Explanation:** Every successful get() moves that key to the back via move_to_end(), so the front of the OrderedDict is always occupied by whichever entry has gone longest without being touched. When put() pushes the cache over capacity, popitem(last=False) removes exactly that front entry — the eviction step — which is why last=False (not the default last=True) is essential here.

## Exercise 14. Compare Two OrderedDicts

**Concept:** order-sensitive equality vs. plain dict equality

**Problem:** Show that OrderedDict equality cares about insertion order, while plain dict equality does not.

**Given:**
```
the same key-value pairs, constructed in different orders
```

**Expected Output:**
```
dict equality: True
OrderedDict equality (different order): False
OrderedDict equality (same order): True
```

**Hint:** Comparing an OrderedDict to a plain dict falls back to order-insensitive dict-style equality.

In [ ]:
from collections import OrderedDict

dict_1 = {"a": 1, "b": 2, "c": 3}
dict_2 = {"c": 3, "a": 1, "b": 2}
print(f"dict equality (different order)      : {dict_1 == dict_2}")

od_1 = OrderedDict([("a", 1), ("b", 2), ("c", 3)])
od_2 = OrderedDict([("c", 3), ("a", 1), ("b", 2)])
print(f"OrderedDict equality (different order): {od_1 == od_2}")

od_3 = OrderedDict([("a", 1), ("b", 2), ("c", 3)])
print(f"OrderedDict equality (same order)     : {od_1 == od_3}")

print(f"OrderedDict vs dict (same data)       : {od_1 == dict_1}")

**Explanation:** Plain dicts consider only key-value pairs for equality, ignoring order entirely. Two OrderedDicts, by contrast, must match both their contents AND their insertion order to be considered equal — od_1 and od_2 hold identical data but in different sequences, so they compare unequal. Comparing an OrderedDict against a plain dict falls back to the plain dict's order-insensitive rule, which can be a surprising asymmetry worth remembering.

## Exercise 15. Reverse an OrderedDict

**Concept:** reversed() on OrderedDict.items()

**Problem:** Iterate an OrderedDict's items from most-recently-inserted back to first-inserted.

**Given:**
```
5 steps in an OrderedDict
```

**Expected Output:**
```
step_5 down to step_1
```

**Hint:** reversed(od.items()) (Python 3.8+) gives a lazy reverse iterator with no data copied.

In [ ]:
from collections import OrderedDict

steps = OrderedDict([
    ("step_1", "Load data"),
    ("step_2", "Clean data"),
    ("step_3", "Analyse"),
    ("step_4", "Visualise"),
    ("step_5", "Export"),
])

print("Forward order:")
for key, value in steps.items():
    print(f"  {key}: {value}")

print("\nReverse order:")
for key, value in reversed(steps.items()):
    print(f"  {key}: {value}")

**Explanation:** reversed() works directly on an OrderedDict's .items(), .keys(), and .values() views (since Python 3.8), producing a lazy reverse iterator rather than building a reversed copy — memory-efficient even for very large dictionaries. This pattern suits undo stacks and audit logs, where the newest entry should naturally appear first.

## Exercise 16. Basic deque Operations

**Concept:** append/appendleft/pop/popleft/extend/extendleft

**Problem:** Demonstrate the core deque operations at both ends.

**Given:**
```
dq = deque([2, 3, 4])
```

**Expected Output:**
```
Sequence of states after append(5), appendleft(1), pop(), popleft(), extend, extendleft
```

**Hint:** extendleft adds items one at a time to the left, which reverses their relative order in the deque.

In [ ]:
from collections import deque

dq = deque([2, 3, 4])
print(f"Initial              : {list(dq)}")

dq.append(5)
print(f"After append(5)      : {list(dq)}")

dq.appendleft(1)
print(f"After appendleft(1)  : {list(dq)}")

right = dq.pop()
print(f"After pop()  -> {right}   : {list(dq)}")

left = dq.popleft()
print(f"After popleft() -> {left} : {list(dq)}")

dq.extend([6, 7])
print(f"After extend([6,7])  : {list(dq)}")

dq.extendleft([0, -1])
print(f"After extendleft([0,-1]): {list(dq)}")

**Explanation:** append/appendleft and pop/popleft all run in O(1), unlike list.insert(0, x) or list.pop(0), which are O(n) because every other element has to shift. extendleft([0, -1]) adds 0 to the left first, then -1 to the left, so -1 ends up as the new leftmost element — the iterable's order gets reversed in the final deque.

## Exercise 17. Implement a Queue (FIFO)

**Concept:** deque.append() + deque.popleft()

**Problem:** Simulate a customer service line: enqueue at the back, serve from the front.

**Given:**
```
a sequence of arrivals and serve calls
```

**Expected Output:**
```
Customers served in the order they arrived
```

**Hint:** `if queue:` is the idiomatic empty check before calling popleft(), since an empty deque is falsy.

In [ ]:
from collections import deque

queue = deque()

def enqueue(customer):
    queue.append(customer)
    print(f"  Arrived : {customer:10s} | Queue: {list(queue)}")

def serve():
    if queue:
        customer = queue.popleft()
        print(f"  Served  : {customer:10s} | Queue: {list(queue)}")
    else:
        print("  No customers waiting.")

print("=== Customer Service Queue ===")
enqueue("Alice")
enqueue("Bob")
enqueue("Charlie")
serve()
enqueue("Diana")
serve()
serve()
serve()
serve()

**Explanation:** append() adds to the right end, and popleft() removes from the left end — using opposite ends is what defines First-In-First-Out ordering. Both run in O(1) on a deque, whereas the list equivalent list.pop(0) would be O(n) since every remaining element must shift left by one position.

## Exercise 18. Implement a Stack (LIFO)

**Concept:** deque.append() + deque.pop() (same end)

**Problem:** Simulate browser back-navigation using a deque as a stack.

**Given:**
```
a sequence of page visits and back-navigation calls
```

**Expected Output:**
```
Most recently visited page is the first one 'gone back' from
```

**Hint:** Using the same end (append and pop, both on the right) is what makes this Last-In-First-Out, unlike the queue's opposite-end approach.

In [ ]:
from collections import deque

history = deque()

def visit(page):
    history.append(page)
    print(f"  Visited : {page:20s} | Current: {history[-1]}")

def go_back():
    if len(history) > 1:
        history.pop()
        print(f"  Back    : {'':20s} | Current: {history[-1]}")
    elif len(history) == 1:
        print(f"  Already at first page: {history[0]}")
    else:
        print("  No history.")

print("=== Browser Navigation Stack ===")
visit("https://home.com")
visit("https://about.com")
visit("https://products.com")
visit("https://contact.com")
go_back()
go_back()
go_back()
go_back()

**Explanation:** history[-1] peeks at the top of the stack (the current page) without removing it. Both append() and pop() operate on the same (right) end here, which is the defining difference from the queue in the previous exercise — that's what makes the most-recently-visited page always the first one removed.

## Exercise 19. Rotating a deque

**Concept:** deque.rotate(n)

**Problem:** Rotate a deque of tasks right by 2, then left by 2, and use rotation for round-robin scheduling.

**Given:**
```
tasks = deque(["Design", "Develop", "Test", "Review", "Deploy"])
```

**Expected Output:**
```
Rotating +2 then -2 restores the original order
```

**Hint:** rotate(n) shifts right for positive n, left for negative n; rotate(-1) repeatedly implements round-robin.

In [ ]:
from collections import deque

tasks = deque(["Design", "Develop", "Test", "Review", "Deploy"])

print(f"Original order   : {list(tasks)}")

tasks.rotate(2)
print(f"After rotate(+2) : {list(tasks)}")

tasks.rotate(-2)
print(f"After rotate(-2) : {list(tasks)}")

print("\nRound-robin task scheduling (5 turns):")
for turn in range(1, 6):
    current_task = tasks[0]
    print(f"  Turn {turn}: processing '{current_task}'")
    tasks.rotate(-1)

**Explanation:** rotate(2) moves the last two elements around to the front, wrapping them; rotate(-2) reverses that exact shift, restoring the original order. Calling rotate(-1) after processing tasks[0] moves the just-handled task to the back and brings the next one to the front — a clean way to implement circular scheduling without any manual index or modulo math.

## Exercise 20. Bounded deque for Recent History

**Concept:** deque(maxlen=N)

**Problem:** Maintain a fixed-size history of the 5 most recent page visits, auto-discarding the oldest.

**Given:**
```
6 page visits added to a deque(maxlen=5)
```

**Expected Output:**
```
After the 6th page is added, the oldest (1st) page is automatically dropped
```

**Hint:** No manual eviction logic is needed — append() on a full maxlen deque drops the opposite end automatically.

In [ ]:
from collections import deque

history = deque(maxlen=5)

pages = [
    "https://home.com",
    "https://about.com",
    "https://products.com",
    "https://blog.com",
    "https://contact.com",
    "https://pricing.com",
]

print(f"Max history size: {history.maxlen}\n")

for page in pages:
    history.append(page)
    print(f"Visited: {page}")
    print(f"History: {list(history)}\n")

**Explanation:** Setting maxlen=5 caps the deque's size permanently. Once full, every new append() silently drops the leftmost (oldest) element before adding the new one at the right — no length checks or manual pops required. Using appendleft() instead would evict from the opposite (right) end; eviction always happens at the end opposite to wherever you're inserting.

## Exercise 21. Create a namedtuple

**Concept:** namedtuple fields, attribute vs. index access

**Problem:** Define a Point namedtuple and access its coordinates by name, by index, and via unpacking.

**Given:**
```
Point(3, 7) and Point(10, 4)
```

**Expected Output:**
```
p1.x: 3, p1[0]: 3; distance between the points calculated
```

**Hint:** p.x and p[0] both work — a namedtuple is a genuine tuple subclass with named fields added on top.

In [ ]:
from collections import namedtuple
import math

Point = namedtuple("Point", ["x", "y"])

p1 = Point(3, 7)
p2 = Point(10, 4)

print(f"p1 - x: {p1.x}, y: {p1.y}")
print(f"p2 - x: {p2.x}, y: {p2.y}")

print(f"p1 via index  - [0]: {p1[0]}, [1]: {p1[1]}")

x, y = p2
print(f"p2 unpacked   - x={x}, y={y}")

distance = math.sqrt((p2.x - p1.x)**2 + (p2.y - p1.y)**2)
print(f"Distance p1 to p2: {distance:.2f}")

**Explanation:** namedtuple("Point", ["x", "y"]) creates a new tuple subclass whose instances support both p.x (readable, preferred in application code) and p[0] (useful when passing to code expecting a plain tuple). Because it's still a genuine tuple, ordinary unpacking (x, y = p2) works exactly as it would on any two-element tuple.

## Exercise 22. namedtuple as a Lightweight Record

**Concept:** filtering a list of namedtuples

**Problem:** Filter a list of Employee namedtuples down to a specific department and compute the average salary.

**Given:**
```
5 Employee records across 3 departments
```

**Expected Output:**
```
All Engineering employees listed with salaries, plus the department average
```

**Hint:** e.department == target_dept reads far more clearly than the index-based e[1] == target_dept.

In [ ]:
from collections import namedtuple

Employee = namedtuple("Employee", ["name", "department", "salary"])

employees = [
    Employee("Alice",   "Engineering", 95000),
    Employee("Bob",     "Marketing",   72000),
    Employee("Charlie", "Engineering", 88000),
    Employee("Diana",   "HR",          65000),
    Employee("Eve",     "Engineering", 102000),
]

target_dept = "Engineering"
print(f"Employees in {target_dept}:")

dept_employees = [e for e in employees if e.department == target_dept]

for emp in dept_employees:
    print(f"  {emp.name:10s} | Salary: £{emp.salary:,}")

print(f"\nTotal in {target_dept}: {len(dept_employees)}")
avg = sum(e.salary for e in dept_employees) / len(dept_employees)
print(f"Average salary     : £{avg:,.0f}")

**Explanation:** Named-field access (e.department) makes the filter condition read like a plain English business rule, unlike the equivalent index-based e[1]. Because namedtuple instances are immutable tuples, the filtered dept_employees list holds direct references to the originals — filtering never copies or duplicates any underlying data.

## Exercise 23. Convert namedtuple to Dictionary

**Concept:** ._asdict() for JSON serialization

**Problem:** Convert a namedtuple to a dict, serialize it as JSON, and reconstruct it back.

**Given:**
```
Product(name="Laptop", category="Electronics", price=999.99, in_stock=True)
```

**Expected Output:**
```
Product as a dict, then as formatted JSON, then reconstructed and confirmed equal to the original
```

**Hint:** json.dumps() can't handle a namedtuple directly — convert with ._asdict() first.

In [ ]:
from collections import namedtuple
import json

Product = namedtuple("Product", ["name", "category", "price", "in_stock"])

product = Product(name="Laptop", category="Electronics",
                  price=999.99, in_stock=True)

product_dict = product._asdict()
print("As dict:")
print(f"  {product_dict}")

product_json = json.dumps(product_dict, indent=2)
print("\nAs JSON:")
print(product_json)

restored = Product(**product_dict)
print(f"\nRestored namedtuple: {restored}")
print(f"Equal to original  : {restored == product}")

**Explanation:** ._asdict() converts the namedtuple's fields into a genuine dict, which json.dumps() can serialize directly — passing the raw namedtuple to json.dumps() would raise a TypeError, since the JSON encoder doesn't recognize named-field tuples. Product(**product_dict) reverses the process, spreading the dict's keys as keyword arguments back into the constructor, completing a clean round trip.

## Exercise 24. Replace a Field Value

**Concept:** ._replace(field=value) for immutable updates

**Problem:** Create modified copies of a namedtuple record without mutating the original.

**Given:**
```
Listing(title="Wireless Headphones", price=79.99, available=True, rating=4.3)
```

**Expected Output:**
```
Two modified copies produced; original remains completely unchanged
```

**Hint:** _replace() always returns a brand-new instance — the original object is never touched.

In [ ]:
from collections import namedtuple

Listing = namedtuple("Listing", ["title", "price", "available", "rating"])

original = Listing(title="Wireless Headphones", price=79.99,
                   available=True, rating=4.3)

print(f"Original  : {original}")

discounted = original._replace(price=59.99)
print(f"Discounted: {discounted}")

updated = original._replace(available=False, rating=4.5)
print(f"Updated   : {updated}")

print(f"\nOriginal unchanged: {original}")
print(f"Same object? {original is discounted}")

**Explanation:** _replace(price=59.99) builds and returns a completely new namedtuple with just that one field changed and everything else copied from original — this is necessary precisely because namedtuples, like all tuples, are immutable and can't be edited in place. Multiple fields can be changed in one call, as updated shows; original is discounted checking object identity confirms False, proving no in-place mutation ever occurred.

## Exercise 25. namedtuple With Default Values

**Concept:** the defaults= parameter

**Problem:** Define a Config namedtuple where some fields are required and others fall back to defaults.

**Given:**
```
Config with required host/port and optional debug/timeout/max_retries
```

**Expected Output:**
```
A fully-specified config and a minimal config, with defaults filling the omitted fields
```

**Hint:** defaults=[...] applies to the rightmost fields first — required fields must come before optional ones in the field list.

In [ ]:
from collections import namedtuple

Config = namedtuple(
    "Config",
    ["host", "port", "debug", "timeout", "max_retries"],
    defaults=[False, 30, 3]
)

print(f"Field defaults: {Config._field_defaults}\n")

full_config = Config(
    host="api.example.com",
    port=8443,
    debug=True,
    timeout=60,
    max_retries=5
)

minimal_config = Config(host="localhost", port=5000)

print(f"Full config   : {full_config}")
print(f"Minimal config: {minimal_config}")

print("\nMinimal config field breakdown:")
for field, value in minimal_config._asdict().items():
    source = "(default)" if field in Config._field_defaults and \
             value == Config._field_defaults[field] else "(provided)"
    print(f"  {field:15s}: {str(value):6s} {source}")

**Explanation:** defaults=[False, 30, 3] supplies fallback values for exactly the last three fields (debug, timeout, max_retries) in order — defaults always fill from the right, so any required (no-default) fields must be listed first. Config._field_defaults exposes which fields have defaults and what they are, which is how the breakdown loop can label each field as provided or defaulted.